In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# 1. Agent
from agents import Agent
agent=Agent(name="Math Tutor",
            instructions="You provide help with math problems. Explain your reasoning at each step and include examples",
            model="gpt-4.1-nano")

In [4]:
# adding more agent
history_agent=Agent(name="history_agent",
                    handoff_description="Specialist agent for historical questions",
                    instructions="You provide assistance with historical queries. Explain important events and context clearly.",
                    model="gpt-4.1-nano")
math_agent=Agent(name="Math Tutor",
                 handoff_description="Specialist agent for Maths questions",
                instructions="You provide help with math problems. Explain your reasoning at each step and include examples",
                model="gpt-4.1-nano")

In [ ]:
triagent=Agent(
    name="Triagent",
    instructions="You determine which agent to use based on the user's homework question",
    handoffs=[history_agent,math_agent],
    model="gpt-4.1-nano"
)

In [10]:
from agents import Runner

async def main():
    result=await Runner.run(triagent,"what is captial of france")
    print(result.final_output)

In [12]:
from agents import GuardrailFunctionOutput
from pydantic import BaseModel

class HomeWorkOutput(BaseModel):
    is_homework:bool
    reasoning:str

guardrail_agent=Agent(name="Guardrail check",instructions="check if the user is asking about homework.",output_type=HomeWorkOutput,model="gpt-4.1-nano")

async def homework_guardrail(ctx,agent,input_data):
    result=await Runner.run(guardrail_agent,input_data,context=ctx.context)
    final_output=result.final_output_as(HomeWorkOutput)
    return GuardrailFunctionOutput(output_info=final_output,tripwire_triggered=not final_output.is_homework)

In [ ]:
from agents import Agent, InputGuardrail, GuardrailFunctionOutput, Runner
from agents.exceptions import InputGuardrailTripwireTriggered
from pydantic import BaseModel
import asyncio

class HomeworkOutput(BaseModel):
    is_homework: bool
    reasoning: str

guardrail_agent = Agent(
    name="Guardrail check",
    instructions="Check if the user is asking about homework.",
    output_type=HomeworkOutput,
)

math_tutor_agent = Agent(
    name="Math Tutor",
    handoff_description="Specialist agent for math questions",
    instructions="You provide help with math problems. Explain your reasoning at each step and include examples",
)

history_tutor_agent = Agent(
    name="History Tutor",
    handoff_description="Specialist agent for historical questions",
    instructions="You provide assistance with historical queries. Explain important events and context clearly.",
)


async def homework_guardrail(ctx, agent, input_data):
    result = await Runner.run(guardrail_agent, input_data, context=ctx.context)
    final_output = result.final_output_as(HomeworkOutput)
    return GuardrailFunctionOutput(
        output_info=final_output,
        tripwire_triggered=not final_output.is_homework,
    )

triage_agent = Agent(
    name="Triage Agent",
    instructions="You determine which agent to use based on the user's homework question",
    handoffs=[history_tutor_agent, math_tutor_agent],
    input_guardrails=[
        InputGuardrail(guardrail_function=homework_guardrail),
    ],
)

async def main():
    try:
        result = await Runner.run(triage_agent, "who was the first president of the united states?")
        print(result.final_output)
    except InputGuardrailTripwireTriggered as e:
        print("Guardrail blocked this input:", e)

    try:
        result = await Runner.run(triage_agent, "What is the meaning of life?")
        print(result.final_output)
    except InputGuardrailTripwireTriggered as e:
        print("Guardrail blocked this input:", e)

if __name__ == "__main__":
    asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop